# Train vs test: conventions and distribution checks

This notebook **does not build new features**. It uses existing CSV columns to:

1. **Convention audit** — column alignment, missing-value patterns, and whether categorical *vocabularies* match between `data/train/` and `data/test/` (including case / formatting differences).
2. **Statistical comparison** — side-by-side summaries and simple tests on distributions of existing numeric and categorical fields, with notes for modeling.

**Paths:** `train_*` under `data/train/`, `test_*` under `data/test/`.

In [1]:
from __future__ import annotations

import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_ROOT = Path("/home/ansar/work/hack-nu-26")
TRAIN = PROJECT_ROOT / "data" / "train"
TEST = PROJECT_ROOT / "data" / "test"

CHUNK_ROWS = 500_000

## 1. Schema: paired files and column names

We expect the same logical tables in train and test, except `train_users.csv` includes the label `churn_status` while `test_users.csv` does not.

In [2]:
pairs = [
    ("users", "train_users.csv", "test_users.csv"),
    ("quizzes", "train_users_quizzes.csv", "test_users_quizzes.csv"),
    ("properties", "train_users_properties.csv", "test_users_properties.csv"),
    ("purchases", "train_users_purchases.csv", "test_users_purchases.csv"),
    ("transaction_attempts", "train_users_transaction_attempts.csv", "test_users_transaction_attempts.csv"),
    ("generations", "train_users_generations.csv", "test_users_generations.csv"),
]

schema_rows = []
for name, tf, vf in pairs:
    ct = pd.read_csv(TRAIN / tf, nrows=0).columns.tolist()
    cv = pd.read_csv(TEST / vf, nrows=0).columns.tolist()
    schema_rows.append(
        {
            "table": name,
            "train_cols": len(ct),
            "test_cols": len(cv),
            "same_order_and_names": ct == cv,
            "only_in_train": sorted(set(ct) - set(cv)),
            "only_in_test": sorted(set(cv) - set(ct)),
        }
    )

schema_df = pd.DataFrame(schema_rows)
display(schema_df[["table", "train_cols", "test_cols", "same_order_and_names"]])
for _, r in schema_df.iterrows():
    if r["only_in_train"] or r["only_in_test"]:
        print(r["table"], "only_in_train:", r["only_in_train"], "only_in_test:", r["only_in_test"])

,table,train_cols,test_cols,same_order_and_names
0,users,3,2,False
1,quizzes,10,10,True
2,properties,5,5,True
3,purchases,6,6,True
4,transaction_attempts,20,20,True
5,generations,12,12,True


users only_in_train: ['churn_status'] only_in_test: []


### 1.1 Known column-name typo (consistent in both splits)

Both generations files use `aspect_ration` (likely meant `aspect_ratio`). No fix applied here — only flagged for interpretation.

## 2. Convention helpers

- **Exact vocabulary:** unique non-null strings as stored.
- **Case-fold map:** group values by `str.strip().casefold()` to detect *Filmmaking* vs *filmmaking* vs *FILMMAKING* appearing as different levels.
- **Only-train / only-test:** string values that never appear in the other split (exact).

In [3]:
def series_vocab(s: pd.Series) -> set[str]:
    s = s.dropna()
    if s.empty:
        return set()
    if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
        return set(s.astype(str).str.strip().replace({"nan": np.nan}).dropna().unique())
    return set(s.unique())


def casefold_groups(values: set[str]) -> dict[str, list[str]]:
    """Map casefold key -> list of original spellings (multiplicity of casing conventions)."""
    groups: dict[str, list[str]] = {}
    for v in sorted(values):
        k = v.strip().casefold()
        groups.setdefault(k, []).append(v)
    return {k: vs for k, vs in groups.items() if len(vs) > 1}


def compare_categorical(train_col: pd.Series, test_col: pd.Series, col_name: str, max_examples: int = 25):
    vt, ve = series_vocab(train_col), series_vocab(test_col)
    only_t = sorted(vt - ve)
    only_e = sorted(ve - vt)
    g_t = casefold_groups(vt)
    g_e = casefold_groups(ve)
    # Cross-split: same casefold in both but different exact strings
    keys_t, keys_e = set(g_t.keys()), set(g_e.keys())
    shared_keys = keys_t & keys_e
    cross_mismatch = []
    for k in sorted(shared_keys):
        st, se = set(g_t[k]), set(g_e[k])
        if st != se:
            cross_mismatch.append((k, sorted(st), sorted(se)))

    out = {
        "column": col_name,
        "n_unique_train": len(vt),
        "n_unique_test": len(ve),
        "n_only_train": len(only_t),
        "n_only_test": len(only_e),
        "intra_train_case_variants": len(g_t),
        "intra_test_case_variants": len(g_e),
        "cross_split_casefold_mismatches": len(cross_mismatch),
    }
    return out, only_t[:max_examples], only_e[:max_examples], cross_mismatch[:max_examples], g_t, g_e


def print_compare_summary(meta, only_t, only_e, cross, label: str):
    print(f"\n=== {label} ===")
    display(pd.DataFrame([meta]))
    if only_t:
        print("Examples only in TRAIN (exact):", only_t)
    if only_e:
        print("Examples only in TEST (exact):", only_e)
    if cross:
        print("Cross-split casefold groups with different spellings (train vs test):")
        for k, st, se in cross[:15]:
            print(f"  key={k!r} train={st} test={se}")

## 3. Load user-level tables (small enough to read fully)

`train_users_transaction_attempts.csv` has **no `user_id`** — rows are keyed by `transaction_id` only (same in test). Convention checks below treat it as a standalone event table.

In [4]:
qu_tr = pd.read_csv(TRAIN / "train_users_quizzes.csv", low_memory=False)
qu_te = pd.read_csv(TEST / "test_users_quizzes.csv", low_memory=False)
pr_tr = pd.read_csv(TRAIN / "train_users_properties.csv")
pr_te = pd.read_csv(TEST / "test_users_properties.csv")

print("Rows:", "quizzes", len(qu_tr), len(qu_te), "properties", len(pr_tr), len(pr_te))

Rows: quizzes 90004 7000 properties 90000 7000


### 3.1 Quizzes — categorical conventions

Expect large **vocabulary drift** on free-text–like fields (e.g. `role`) between train (many distinct strings) and test (fewer, more standardized levels). `first_feature` often differs by **naming style** (e.g. Title Case vs kebab-case).

In [5]:
# Include `string` dtypes — `dtype == object` misses pandas StringDtype / string[pyarrow], leaving an empty list.
quiz_cat_cols = [
    c
    for c in qu_tr.select_dtypes(include=["object", "string"]).columns
    if c not in ("Unnamed: 0", "user_id")
]
quiz_summaries = []
for c in quiz_cat_cols:
    meta, ot, oe, cross, _, _ = compare_categorical(qu_tr[c], qu_te[c], c)
    quiz_summaries.append(meta)
    print_compare_summary(meta, ot, oe, cross, f"quizzes.{c}")

quiz_summary_df = pd.DataFrame(quiz_summaries)
if quiz_summary_df.empty:
    print("No string/object quiz columns found; check dtypes:", qu_tr.dtypes)
else:
    display(quiz_summary_df.sort_values("n_only_train", ascending=False))


=== quizzes.source ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,source,533,23,515,5,38,0,0


Examples only in TRAIN (exact): ['$10K Digital Asset', ',', ',b', ',drxfg', ',n', '-', '.', '..', '...', '.kg', '/', '//////', '0', '00', '1', '111', '1111', '12', '1233', '123321', '1234', '2', '234к5ц', '4', '58']
Examples only in TEST (exact): ['Alicia Lyttle', 'My friend', 'My team', 'of', 'ы']

=== quizzes.flow_type ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,flow_type,3,0,3,0,0,0,0


Examples only in TRAIN (exact): ['invited', 'personal', 'team']

=== quizzes.team_size ===


...

,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,first_feature,21,19,2,0,1,1,0


Examples only in TRAIN (exact): ['Image Generation', 'Video Generation']

=== quizzes.role ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,role,373,15,362,4,25,0,0


Examples only in TRAIN (exact): ['"', '$$$', "'", ',', ',,', ',hjbh', '-', '--', '.', '..', '...', '.......', '/', '//////', '0', '1', '111', '1111', '12', '12312', '2 4s in a 4 door', '333', '3d', '65', '768uty']
Examples only in TEST (exact): ['fedfsd', 'freelancer', 't', 'о']


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,source,533,23,515,5,38,0,0
7,role,373,15,362,4,25,0,0
1,flow_type,3,0,3,0,0,0,0
6,first_feature,21,19,2,0,1,1,0
4,usage_plan,7,6,1,0,0,0,0
2,team_size,13,13,0,0,0,0,0
3,experience,4,4,0,0,0,0,0
5,frustration,12,12,0,0,1,1,0


### 3.2 Properties — `subscription_plan`, `country_code`, dates

Compare plan labels and country codes. `subscription_start_date` is parsed only for range / missing checks — no new features.

In [6]:
for c in ["subscription_plan", "country_code"]:
    meta, ot, oe, cross, _, _ = compare_categorical(pr_tr[c], pr_te[c], c)
    print_compare_summary(meta, ot, oe, cross, f"properties.{c}")

for name, s_tr, s_te in [
    ("properties.subscription_start_date", pr_tr["subscription_start_date"], pr_te["subscription_start_date"]),
]:
    dtr = pd.to_datetime(s_tr, utc=True, errors="coerce")
    dte = pd.to_datetime(s_te, utc=True, errors="coerce")
    print(f"\n{name} parse null rate train {dtr.isna().mean():.4f} test {dte.isna().mean():.4f}")
    print("  train min/max", dtr.min(), dtr.max())
    print("  test min/max", dte.min(), dte.max())


=== properties.subscription_plan ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,subscription_plan,4,4,0,0,0,0,0



=== properties.country_code ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,country_code,218,153,67,2,1,0,0


Examples only in TRAIN (exact): ['AF', 'AG', 'AQ', 'BM', 'BO', 'BQ', 'BT', 'BV', 'BW', 'BZ', 'CD', 'CF', 'CG', 'CK', 'CV', 'EH', 'ER', 'FK', 'FO', 'GD', 'GG', 'GL', 'GM', 'GN', 'GP']
Examples only in TEST (exact): ['BL', 'MW']

properties.subscription_start_date parse null rate train 0.0000 test 0.0000
  train min/max 1067-08-26 00:00:07+00:00 1067-11-09 23:59:45+00:00
  test min/max 1067-11-10 00:09:15+00:00 1067-11-23 23:56:03+00:00


## 4. Purchases and transaction attempts (event-level)

Row counts and file sizes differ greatly; we compare **marginal** distributions of columns within each file pair.

In [7]:
pu_tr = pd.read_csv(TRAIN / "train_users_purchases.csv")
pu_te = pd.read_csv(TEST / "test_users_purchases.csv")
ta_tr = pd.read_csv(TRAIN / "train_users_transaction_attempts.csv")
ta_te = pd.read_csv(TEST / "test_users_transaction_attempts.csv")

print("purchases rows", len(pu_tr), len(pu_te))
print("transaction_attempts rows", len(ta_tr), len(ta_te))

purchases rows 96424 8825
transaction_attempts rows 178098 11822


/tmp/ipykernel_58313/3541134771.py:3: DtypeWarning: Columns (0: is_prepaid, 1: is_virtual, 2: is_business) have mixed types. Specify dtype option on import or set low_memory=False.
  ta_tr = pd.read_csv(TRAIN / "train_users_transaction_attempts.csv")


In [8]:
for c in ["purchase_type"]:
    meta, ot, oe, cross, _, _ = compare_categorical(pu_tr[c], pu_te[c], c)
    print_compare_summary(meta, ot, oe, cross, f"purchases.{c}")

ta_cat = [
    c
    for c in ta_tr.select_dtypes(include=["object", "string"]).columns
    if c not in ("bank_name", "transaction_id", "failure_code")
]
# bank_name is high cardinality; sample convention on a few other columns
for c in ta_cat:
    meta, ot, oe, cross, _, _ = compare_categorical(ta_tr[c], ta_te[c], c)
    if meta["n_only_train"] or meta["n_only_test"] or meta["cross_split_casefold_mismatches"]:
        print_compare_summary(meta, ot, oe, cross, f"transaction_attempts.{c}")

meta, ot, oe, cross, _, _ = compare_categorical(ta_tr["failure_code"].fillna(""), ta_te["failure_code"].fillna(""), "failure_code")
print_compare_summary(meta, ot, oe, cross, "transaction_attempts.failure_code (empty string = NaN)")


=== purchases.purchase_type ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,purchase_type,6,5,1,0,0,0,0


Examples only in TRAIN (exact): ['Reactivation']

=== transaction_attempts.transaction_time ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,transaction_time,175109,11720,175074,11685,0,0,0


Examples only in TRAIN (exact): ['1067-08-26 00:00:07+00:00', '1067-08-26 00:01:30+00:00', '1067-08-26 00:01:56+00:00', '1067-08-26 00:02:43+00:00', '1067-08-26 00:06:31+00:00', '1067-08-26 00:08:04+00:00', '1067-08-26 00:08:39+00:00', '1067-08-26 00:09:50+00:00', '1067-08-26 00:13:05+00:00', '1067-08-26 00:13:51+00:00', '1067-08-26 00:14:48+00:00', '1067-08-26 00:16:49+00:00', '1067-08-26 00:19:48+00:00', '1067-08-26 00:20:16+00:00', '1067-08-26 00:20:22+00:00', '1067-08-26 00:21:56+00:00', '1067-08-26 00:31:11+00:00', '1067-08-26 00:35:05+00:00', '1067-08-26 00:46:26+00:00', '1067-08-26 00:56:16+00:00', '1067-08-26 00:57:00+00:00', '1067-08-26 00:57:08+00:00', '1067-08-26 00:57:47+00:00', '1067-08-26 00:58:12+00:00', '1067-08-26 00:58:28+00:00']
Examples only in TEST (exact): ['1067-11-10 00:09:15+00:00', '1067-11-10 00:13:48+00:00', '1067-11-10 00:16:17+00:00', '1067-11-10 00:16:54+00:00', '1067-11-10 00:18:02+00:00', '1067-11-10 00:20:37+00:00', '1067-11-10 00:20:47+00:00', '1067-1

...

,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,card_country,181,134,48,1,0,0,0


Examples only in TRAIN (exact): ['ad', 'af', 'ag', 'ao', 'bf', 'bj', 'bm', 'bo', 'bq', 'bt', 'bw', 'bz', 'cd', 'cg', 'cm', 'cv', 'dj', 'dm', 'fj', 'gd', 'gm', 'gn', 'gq', 'gu', 'gw']
Examples only in TEST (exact): ['sb']

=== transaction_attempts.bank_country ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,bank_country,174,126,49,1,1,0,0


Examples only in TRAIN (exact): ['Afghanistan', 'Andorra', 'Antigua and Barbuda', 'BOLIVIA', 'BRITISH VIRGIN ISLANDS', 'Benin', 'Bermuda', 'Bhutan', 'Botswana', 'Burkina Faso', 'Cameroon', 'Cape Verde', 'Chad', 'Congo', 'Czech Republic', 'European Union', 'Fiji', 'Gabon', 'Gibraltar', 'Grenada', 'Guinea', 'Guyana', 'IVORY COAST', 'Iceland', 'Lesotho']
Examples only in TEST (exact): ['Libya']

=== transaction_attempts.failure_code (empty string = NaN) ===


,column,n_unique_train,n_unique_test,n_only_train,n_only_test,intra_train_case_variants,intra_test_case_variants,cross_split_casefold_mismatches
0,failure_code,8,8,0,0,0,0,0


## 5. Generations — chunked read (train file is multi-GB)

We accumulate **value counts** for categoricals and **running moments** for numerics without loading the full train file.

Test file is read fully (fits in memory).

In [9]:
gen_cat = ["status", "generation_type", "resolution", "aspect_ration"]
gen_num = ["credit_cost", "duration"]


def accumulate_counts(path: Path, cols: list[str], chunksize: int) -> dict[str, Counter]:
    ctr = {c: Counter() for c in cols}
    for chunk in pd.read_csv(path, usecols=cols, chunksize=chunksize):
        for c in cols:
            ctr[c].update(chunk[c].dropna().astype(str).tolist())
    return ctr


def accumulate_numeric_stats(path: Path, cols: list[str], chunksize: int) -> dict[str, dict]:
    """Per-column Welford variance; one pass per column to keep state correct."""
    out: dict[str, dict] = {}
    for c in cols:
        n = 0
        mean = 0.0
        m2 = 0.0
        vmin = np.inf
        vmax = -np.inf
        nnan = 0
        for chunk in pd.read_csv(path, usecols=[c], chunksize=chunksize):
            x = pd.to_numeric(chunk[c], errors="coerce")
            nnan += int(x.isna().sum())
            xv = x.dropna().to_numpy(dtype=float)
            for v in xv:
                n += 1
                delta = v - mean
                mean += delta / n
                m2 += delta * (v - mean)
                vmin = min(vmin, float(v))
                vmax = max(vmax, float(v))
        var = (m2 / (n - 1)) if n > 1 else np.nan
        out[c] = {"n": n, "mean": mean, "var": var, "min": vmin, "max": vmax, "nnan": nnan}
    return out


print("Accumulating train generations (chunked)...")
tr_counts = accumulate_counts(TRAIN / "train_users_generations.csv", gen_cat, CHUNK_ROWS)
print("Train numeric scan (one pass per column)...")
tr_num_stats = accumulate_numeric_stats(TRAIN / "train_users_generations.csv", gen_num, CHUNK_ROWS)
c0 = gen_num[0]
tr_row_count = tr_num_stats[c0]["n"] + tr_num_stats[c0]["nnan"]

ge_te = pd.read_csv(TEST / "test_users_generations.csv")
te_counts = {c: Counter(ge_te[c].dropna().astype(str)) for c in gen_cat}

print("Train generations rows (line count - 1):", tr_row_count)
print("Test generations rows:", len(ge_te))

Accumulating train generations (chunked)...
Train numeric scan (one pass per column)...


/tmp/ipykernel_58313/3550961954.py:46: DtypeWarning: Columns (0: resolution) have mixed types. Specify dtype option on import or set low_memory=False.
  ge_te = pd.read_csv(TEST / "test_users_generations.csv")


Train generations rows (line count - 1): 28474033
Test generations rows: 1078213


In [10]:
def top_k(counter: Counter, k: int = 15):
    return counter.most_common(k)


def compare_counters(ct: Counter, ce: Counter, name: str, k: int = 15):
    st, se = set(ct), set(ce)
    only_t = sorted(st - se)[:20]
    only_e = sorted(se - st)[:20]
    print(f"\n--- {name} ---")
    print("unique train", len(st), "unique test", len(se), "only_train", len(st - se), "only_test", len(se - st))
    if only_t:
        print("examples only train:", only_t)
    if only_e:
        print("examples only test:", only_e)
    tot_t = sum(ct.values())
    tot_e = sum(ce.values())
    keys = sorted(set(ct) | set(ce))
    rows = []
    for key in keys:
        pt = ct.get(key, 0) / tot_t if tot_t else 0
        pe = ce.get(key, 0) / tot_e if tot_e else 0
        rows.append((key, pt, pe, abs(pt - pe)))
    rows.sort(key=lambda x: -x[3])
    display(pd.DataFrame(rows[:k], columns=["value", "p_train", "p_test", "abs_diff"]))


for c in gen_cat:
    compare_counters(tr_counts[c], te_counts[c], f"generations.{c}")


--- generations.status ---
unique train 7 unique test 7 only_train 0 only_test 0


,value,p_train,p_test,abs_diff
0,nsfw,0.085044,0.059307,0.025737
1,failed,0.025624,0.042439,0.016815
2,completed,0.884203,0.892610,0.008408
3,canceled,0.005058,0.005470,0.000412
4,queued,0.000045,0.000156,0.000110
5,in_progress,0.000020,0.000015,0.000005
6,waiting,0.000006,0.000003,0.000004



--- generations.generation_type ---
unique train 22 unique test 21 only_train 1 only_test 0
examples only train: ['video_model_4']


,value,p_train,p_test,abs_diff
0,image_model_1,0.567880,0.392821,0.175059
1,image_model_9,0.003839,0.132041,0.128202
2,image_model_2,0.085028,0.188431,0.103403
3,video_model_8,0.008046,0.062883,0.054837
4,image_model_4,0.051107,0.023859,0.027248
5,video_model_2,0.035627,0.013453,0.022174
6,video_model_1,0.037037,0.015931,0.021106
7,video_model_4,0.012516,0.000000,0.012516
8,image_model_6,0.018862,0.029544,0.010682
9,video_model_7,0.011443,0.002042,0.009401



--- generations.resolution ---
unique train 11 unique test 11 only_train 0 only_test 0


,value,p_train,p_test,abs_diff
0,1k,0.241245,0.260219,0.018975
1,4k,0.179121,0.160628,0.018494
2,720p,0.038844,0.057029,0.018185
3,720,0.009178,0.002742,0.006436
4,1080,0.008698,0.003280,0.005418
5,480,0.005496,0.000917,0.004580
6,480p,0.007032,0.003786,0.003246
7,2k,0.498173,0.501123,0.002949
8,768,0.003852,0.003043,0.000809
9,1080p,0.007755,0.007029,0.000727



--- generations.aspect_ration ---
unique train 13 unique test 12 only_train 1 only_test 0
examples only train: ['2:1']


,value,p_train,p_test,abs_diff
0,3:4,0.124455,0.255140,0.130686
1,1:1,0.186139,0.083280,0.102859
2,auto,0.080520,0.032491,0.048030
3,16:9,0.171173,0.206725,0.035552
4,9:16,0.284356,0.311178,0.026822
5,2:3,0.042987,0.021604,0.021383
6,21:9,0.031840,0.021209,0.010631
7,4:5,0.036397,0.029621,0.006776
8,5:4,0.004991,0.002099,0.002892
9,3:2,0.013605,0.011366,0.002239


In [11]:
print("TRAIN generations numeric (Welford on all rows in file):")
for c in gen_num:
    s = tr_num_stats[c]
    std = np.sqrt(s["var"]) if s["var"] == s["var"] else np.nan
    print(
        c,
        "mean",
        s["mean"],
        "std",
        std,
        "min",
        s["min"],
        "max",
        s["max"],
        "nan_count",
        s["nnan"],
        "finite_n",
        s["n"],
    )

print("\nTEST generations numeric:")
for c in gen_num:
    x = pd.to_numeric(ge_te[c], errors="coerce")
    print(c, "mean", x.mean(), "std", x.std(), "min", x.min(), "max", x.max(), "nan_count", x.isna().sum())

TRAIN generations numeric (Welford on all rows in file):
credit_cost mean 1134.9626795166098 std 1410.779467626419 min 0.0 max 32186.0 nan_count 23837973 finite_n 4636060
duration mean 7.785138643413489 std 3.838045110468288 min 0.0081173333333333 max 214.576 nan_count 23521051 finite_n 4952982

TEST generations numeric:
credit_cost mean 1335.7865634619407 std 1145.9509380492384 min 0.0 max 17500.0 nan_count 910908
duration mean 8.085214250842721 std 4.189510776047875 min 1.0 max 40.727732 nan_count 910008


## 6. Statistical comparison — user-level proportions

Chi-square **homogeneity** on shared categories (quizzes / properties) where we align categories that exist in **both** splits; rare categories are grouped into `__other__` to stabilize counts.

This answers: *if we treat the two files as two samples, do column marginals look like the same multinomial?* — **not** a formal test of i.i.d. sampling, but a useful drift screen.

In [12]:
def homogeneity_chi2(s_train: pd.Series, s_test: pd.Series, top_n: int = 40):
    """Chi-square test of homogeneity; keep `top_n` joint-frequency levels, bucket the rest as __other__."""
    vc_tr = s_train.fillna("__na__").astype(str).value_counts()
    vc_te = s_test.fillna("__na__").astype(str).value_counts()
    combined = vc_tr.add(vc_te, fill_value=0)
    top_keys = list(combined.nlargest(top_n).index)
    top_set = set(top_keys)

    def bucket(vc: pd.Series) -> pd.Series:
        head = vc.reindex(top_keys).fillna(0)
        tail = float(vc.loc[vc.index.difference(top_set)].sum()) if len(vc) else 0.0
        if tail > 0:
            head["__other__"] = tail
        return head

    a = bucket(vc_tr)
    b = bucket(vc_te)
    idx = sorted(set(a.index) | set(b.index))
    obs = np.array([a.reindex(idx, fill_value=0).values, b.reindex(idx, fill_value=0).values])
    chi2, p, dof, _ = stats.chi2_contingency(obs)
    return {"chi2": float(chi2), "p_value": float(p), "dof": int(dof), "n_levels": len(idx)}


for col in ["source", "flow_type", "team_size", "experience", "usage_plan", "frustration", "first_feature", "role"]:
    r = homogeneity_chi2(qu_tr[col], qu_te[col])
    print(f"quizzes.{col}", r)

for col in ["subscription_plan", "country_code"]:
    r = homogeneity_chi2(pr_tr[col], pr_te[col])
    print(f"properties.{col}", r)

quizzes.source {'chi2': 1922.8141996669856, 'p_value': 0.0, 'dof': 40, 'n_levels': 41}
quizzes.flow_type {'chi2': 3.0344184981340714, 'p_value': 0.38634890335986893, 'dof': 3, 'n_levels': 4}
quizzes.team_size {'chi2': 1985.8937313522129, 'p_value': 0.0, 'dof': 13, 'n_levels': 14}
quizzes.experience {'chi2': 2447.6992999792838, 'p_value': 0.0, 'dof': 4, 'n_levels': 5}
quizzes.usage_plan {'chi2': 2555.331358413575, 'p_value': 0.0, 'dof': 7, 'n_levels': 8}
quizzes.frustration {'chi2': 6370.769929042732, 'p_value': 0.0, 'dof': 12, 'n_levels': 13}
quizzes.first_feature {'chi2': 3456.9805785854555, 'p_value': 0.0, 'dof': 21, 'n_levels': 22}
quizzes.role {'chi2': 1643.4785861339155, 'p_value': 0.0, 'dof': 40, 'n_levels': 41}
properties.subscription_plan {'chi2': 2819.8968417260044, 'p_value': 0.0, 'dof': 3, 'n_levels': 4}
properties.country_code {'chi2': 674.4462354613677, 'p_value': 3.2832223633814833e-116, 'dof': 40, 'n_levels': 41}


In [13]:
def ks_two_sample(a: np.ndarray, b: np.ndarray):
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan
    res = stats.ks_2samp(a, b)
    return res.statistic, res.pvalue


pa_tr = pd.to_numeric(pu_tr["purchase_amount_dollars"], errors="coerce")
pa_te = pd.to_numeric(pu_te["purchase_amount_dollars"], errors="coerce")
stat, p = ks_two_sample(pa_tr.to_numpy(), pa_te.to_numpy())
print("KS purchase_amount_dollars", "D", stat, "p", p)

ta_amt_tr = pd.to_numeric(ta_tr["amount_in_usd"], errors="coerce")
ta_amt_te = pd.to_numeric(ta_te["amount_in_usd"], errors="coerce")
stat, p = ks_two_sample(ta_amt_tr.to_numpy(), ta_amt_te.to_numpy())
print("KS transaction_attempts amount_in_usd", "D", stat, "p", p)

KS purchase_amount_dollars D 0.19815368571622638 p 1.1314419024316359e-278
KS transaction_attempts amount_in_usd D 0.19045071281307857 p 0.0


## 7. Takeaways for training

Run the cells above for exact counts and p-values. Qualitative points that already show up in this data:

1. **Label only in train:** `churn_status` exists only in `train_users.csv`; `test_users.csv` is `user_id` only.
2. **Quizzes `role`:** train often has **hundreds** of distinct `role` strings vs **few** in test — treat as high-cardinality / open vocabulary in train and **do not assume** test labels are a subset of the same spelling set. Hashing, frequency bucketing, or strong regularization helps; expect **label / token drift**.
3. **`first_feature`:** train tends toward **Title Case** labels (e.g. `Video Generation`); test often uses **kebab-case** slugs (e.g. `video-creation`). Same semantic idea can appear as **different strings** across splits — use normalization or treat as categorical with large unknown bucket.
4. **Properties `subscription_plan`:** snapshot mix can differ (e.g. more `Higgsfield Pro` in train vs `Higgsfield Creator` / `Ultimate` in test in head rows) — confirm with the contingency / chi-square output.
5. **Purchases:** `purchase_amount_dollars` can have very different scales (small credit packs vs large subscription amounts) — robust scaling or transforms reduce sensitivity to heavy tails.
6. **Generations:** train file is **much larger** than test; marginal distributions of `status`, `generation_type`, `resolution` may differ — compare the printed tables. Chunked train scan is slow but avoids loading 5GB+ into RAM.
7. **Transaction attempts:** no `user_id` in these CSVs — user-level use requires joining via `transaction_id` to purchases (as in your other work).

---

This notebook only **reads** existing columns; it does not add engineered features.